In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, Model, optimizers
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# INPUTS (The clean files you generated)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy") # Raw Payload (N, 10, 784)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy") # Stats (N, 137)

# OUTPUT (Overwriting the weights file)
WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Params
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.0001
TEMPERATURE = 0.1
LATENT_DIM = 128
PROJECTION_DIM = 64

def load_data():
    print("--- Loading Clean Pre-training Data ---")
    if not os.path.exists(X_VIEW1_PATH):
        print(f"FATAL: {X_VIEW1_PATH} not found. Run the Data Prep script first.")
        return None, None

    X1 = np.load(X_VIEW1_PATH).astype('float32')
    X2 = np.load(X_VIEW2_PATH).astype('float32')

    print(f"View 1 (Raw Payload): {X1.shape}")
    print(f"View 2 (Statistics):  {X2.shape}")

    # Normalize Stats
    scaler = StandardScaler()
    X2 = scaler.fit_transform(X2)

    return X1, X2

# --- CNN ARCHITECTURE (Matches 10x784 Input) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape: Treat 10 packets * 784 bytes as one long sequence
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs) # (7840, 1)

    # 2. CNN Block
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    # Flatten -> Dense
    x = layers.Flatten()(x)

    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(PROJECTION_DIM, activation='relu', name="projection")(h)
    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z)

    return Model(inputs, [h, z], name="CNN_Encoder")

# --- MLP ARCHITECTURE (Fixed Typo Here) ---
def get_mlp_encoder(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.Dropout(0.3)(x)

    # Representation Layer (h)
    h = layers.Dense(LATENT_DIM, activation='relu')(x)

    # Projection Layer (z)
    # FIX: Passed 'h' directly, removed incorrect 'h_layer=' keyword
    z = layers.Dense(PROJECTION_DIM, activation='relu')(h)

    z = layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(z)
    return Model(inputs, [h, z], name="MLP_Encoder")

# --- HMVCL MODEL ---
class HMVCL(Model):
    def __init__(self, cnn_encoder, mlp_encoder, temperature=0.1):
        super(HMVCL, self).__init__()
        self.cnn_encoder = cnn_encoder
        self.mlp_encoder = mlp_encoder
        self.temperature = temperature

    def compile(self, optimizer, loss_fn):
        super(HMVCL, self).compile()
        self.optimizer = optimizer
        self.loss_fn = loss_fn

    def train_step(self, data):
        view1, view2 = data
        with tf.GradientTape() as tape:
            # Forward pass through both encoders
            _, z1 = self.cnn_encoder(view1, training=True)
            _, z2 = self.mlp_encoder(view2, training=True)
            # Calculate Contrastive Loss
            loss = self.loss_fn(z1, z2, self.temperature)

        # Update weights for BOTH networks
        trainable_vars = self.cnn_encoder.trainable_variables + self.mlp_encoder.trainable_variables
        grads = tape.gradient(loss, trainable_vars)
        self.optimizer.apply_gradients(zip(grads, trainable_vars))
        return {"loss": loss}

def nt_xent_loss(z1, z2, temperature):
    batch_size = tf.shape(z1)[0]
    z = tf.concat([z1, z2], axis=0)
    sim_matrix = tf.matmul(z, z, transpose_b=True) / temperature

    mask = tf.eye(2 * batch_size, dtype=tf.bool)
    sim_matrix = tf.where(mask, -1e9, sim_matrix)

    labels = tf.concat([tf.range(batch_size) + batch_size, tf.range(batch_size)], axis=0)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, sim_matrix, from_logits=True)
    return tf.reduce_mean(loss)

def main():
    # 1. Load Clean Data
    X1, X2 = load_data()
    if X1 is None: return

    # 2. Prepare Dataset
    dataset = tf.data.Dataset.from_tensor_slices((X1, X2))
    dataset = dataset.shuffle(1024).batch(BATCH_SIZE, drop_remainder=True)

    # 3. Initialize Models
    cnn = get_cnn_encoder((10, 784))
    mlp = get_mlp_encoder(X2.shape[1])

    model = HMVCL(cnn, mlp, temperature=TEMPERATURE)
    model.compile(optimizer=optimizers.Adam(LEARNING_RATE), loss_fn=nt_xent_loss)

    # 4. Train
    print("\n--- Starting Corrected Pre-Training ---")
    model.fit(dataset, epochs=EPOCHS)

    # 5. Save Valid Weights
    print(f"Saving CORRECTED weights to {WEIGHTS_PATH}")
    model.cnn_encoder.save_weights(WEIGHTS_PATH)
    print("Done. You can now run the t-SNE or Fine-tuning scripts.")

if __name__ == "__main__":
    main()